In [1]:
import torch
import torch.nn as nn

In [2]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
neurons = 4

In [3]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
# model initialization
model = MultiLayerPerceptron(input_size=x.shape[-1], output_size=y.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

layers.0.weight: tensor([[ 0.4354, -0.2535,  0.0952, -0.0090, -0.2903],
        [-0.4435, -0.3442,  0.4242, -0.1075, -0.3538],
        [-0.2831,  0.1902, -0.0170, -0.3465, -0.1198],
        [ 0.0775,  0.0072,  0.1666,  0.2181, -0.1029]])
layers.0.bias: tensor([-0.4170, -0.2066, -0.1619, -0.0887])
layers.2.weight: tensor([[-0.4061, -0.0743, -0.1471,  0.1568]])
layers.2.bias: tensor([-0.2408])


In [5]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.6465063691139221
layers.0.weight.grad: tensor([[-0.2590, -0.2078, -0.1567, -0.1055, -0.0543],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0856,  0.0341, -0.0608, -0.1049, -0.1938]])
layers.0.bias.grad: tensor([-0.0398,  0.0000,  0.0000, -0.0052])
layers.2.weight.grad: tensor([[ 0.1026,  0.0000,  0.0000, -0.0364]])
layers.2.bias.grad: tensor([-0.0333])
epoch 50 loss: 0.15122807025909424
layers.0.weight.grad: tensor([[-0.0695, -0.0156, -0.0185,  0.0493,  0.0074],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0404, -0.0043, -0.0086, -0.0632, -0.0398]])
layers.0.bias.grad: tensor([-0.0255,  0.0000,  0.0000,  0.0116])
layers.2.weight.grad: tensor([[ 0.0698,  0.0000,  0.0000, -0.1119]])
layers.2.bias.grad: tensor([0.0198])
epoch 100 loss: 0.05301644280552864
layers.0.weight.grad: tensor([[-0.0278, -0.0155,  0.0001,  0.0137, 

In [6]:
# prediction
with torch.inference_mode():
    z = model(x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
